In [1]:
from pathlib import Path
import json
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from collections import Counter

c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CHUNKS_PATH = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\unified_semantic_chunks\unified_chunks.json"
)
VECTOR_STORE = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store"
)
VECTOR_STORE.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# How many times to duplicate high-value chunks for retrieval boosting
OPERATIONAL_BOOST = 2  # WMS reference chunks embedded N times
SCHEMA_BOOST      = 1  # Table schema chunks (no duplication needed)

print(f"📂 Chunks path:   {CHUNKS_PATH}")
print(f"📂 Vector store:  {VECTOR_STORE}")

📂 Chunks path:   C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\unified_semantic_chunks\unified_chunks.json
📂 Vector store:  C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store


In [3]:
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

# Filter empty text
chunks = [c for c in all_chunks if c.get("text", "").strip()]
skipped = len(all_chunks) - len(chunks)

print(f"✅ Loaded  : {len(all_chunks)} total chunks")
print(f"🔍 Valid   : {len(chunks)} non-empty chunks")
print(f"⏭️  Skipped : {skipped} empty chunks")

# Show chunk type breakdown
type_counts = Counter(c.get("metadata", {}).get("chunk_type", "unknown") for c in chunks)
print(f"\n📊 Chunk types:")
for k, v in type_counts.most_common():
    print(f"   {k:<35} : {v}")

✅ Loaded  : 627 total chunks
🔍 Valid   : 627 non-empty chunks
⏭️  Skipped : 0 empty chunks

📊 Chunk types:
   text_prose                          : 470
   wms_procedure                       : 28
   text_table                          : 25
   schema_overview                     : 18
   wms_overview                        : 18
   wms_join_logic                      : 18
   wms_safety_rules                    : 18
   schema_core_columns                 : 17
   schema_extra_columns                : 15


In [4]:
def detect_document_type(chunk: dict) -> str:
    """
    Detects chunk type using chunk_type metadata first (set during chunking),
    then falls back to structured_data inspection.
    This is more reliable than re-inferring from structured data.
    """
    # Prefer explicit chunk_type set during chunking
    chunk_type = chunk.get("metadata", {}).get("chunk_type", "")

    if chunk_type in ("schema_overview", "schema_core_columns", "schema_extra_columns"):
        return "TABLE_SCHEMA"

    if chunk_type in ("wms_overview", "wms_join_logic", "wms_procedure", "wms_safety_rules"):
        return "OPERATIONAL_REFERENCE"

    if chunk_type in ("text_prose", "text_table"):
        return "TEXT"

    # Fallback: inspect structured_data
    structured = chunk.get("structured_data")
    if isinstance(structured, dict):
        if "columns" in structured:
            return "TABLE_SCHEMA"
        if "procedures" in structured or "core_tables" in structured:
            return "OPERATIONAL_REFERENCE"

    return "TEXT"

In [5]:
def enrich_text(chunk: dict) -> str:
    """
    Prepends structured metadata to the chunk text before embedding.
    This improves semantic search by giving the embedding model
    richer context about what the chunk represents.

    Key fix: TABLE_SCHEMA uses FK data from column objects
    (matching the actual JSON structure), not a top-level foreign_keys array.
    """
    text      = chunk["text"]
    metadata  = chunk.get("metadata", {})
    structured = chunk.get("structured_data")
    doc_type  = detect_document_type(chunk)
    chunk_type = metadata.get("chunk_type", "")

    prefix = []

    # --- Universal fields ---
    if metadata.get("category"):
        prefix.append(f"CATEGORY: {metadata['category']}")
    if metadata.get("source"):
        prefix.append(f"SOURCE: {metadata['source']}")
    prefix.append(f"DOCUMENT TYPE: {doc_type}")

    # --- TABLE_SCHEMA enrichment ---
    if doc_type == "TABLE_SCHEMA":
        table_name = metadata.get("table_name", "")
        if table_name:
            prefix.append(f"TABLE NAME: {table_name}")

        # FK relationships (read from column objects — matches your JSON structure)
        related = metadata.get("related_tables", [])
        if related:
            prefix.append(f"RELATED TABLES: {', '.join(related)}")

        # Only add column names on overview chunk (not repeated on every sub-chunk)
        if chunk_type == "schema_overview" and isinstance(structured, dict):
            columns  = structured.get("columns", [])
            fk_cols  = [c for c in columns if c.get("is_foreign_key")]
            pk       = structured.get("primary_key", "N/A")
            prefix.append(f"PRIMARY KEY: {pk}")
            if fk_cols:
                fk_summary = ", ".join(
                    f"{c['name']}→{c.get('references_table','?')}"
                    for c in fk_cols
                )
                prefix.append(f"FOREIGN KEYS: {fk_summary}")

    # --- OPERATIONAL_REFERENCE enrichment ---
    if doc_type == "OPERATIONAL_REFERENCE":
        prefix.append("CONTAINS VERIFIED OPERATIONAL SQL PROCEDURES")

        # Procedure-level enrichment
        if chunk_type == "wms_procedure" and isinstance(structured, dict):
            proc_name = structured.get("procedure_name", "")
            if proc_name:
                prefix.append(f"PROCEDURE: {proc_name}")
            biz_logic = structured.get("business_logic", "")
            if biz_logic:
                prefix.append(f"BUSINESS LOGIC: {biz_logic}")
            access = structured.get("access_level", "")
            if access:
                prefix.append(f"ACCESS LEVEL: {access}")

        # Join logic enrichment
        if chunk_type == "wms_join_logic" and isinstance(structured, dict):
            inbound  = structured.get("inbound", {})
            outbound = structured.get("outbound", {})
            if inbound:
                prefix.append(f"INBOUND JOIN KEYS: {', '.join(inbound.get('primary_keys', []))}")
            if outbound:
                prefix.append(f"OUTBOUND JOIN KEYS: {', '.join(outbound.get('primary_keys', []))}")

        # Document-level table list
        related = metadata.get("related_tables", [])
        if related:
            prefix.append(f"TABLES INVOLVED: {', '.join(related)}")

        # Keyword boosting for common WMS query terms
        prefix.append(
            "KEYWORDS: reverse, resend, reset, grn, receipt, order, "
            "movement, mission, loading, stock, inbound, outbound"
        )

    # --- TEXT enrichment ---
    if doc_type == "TEXT":
        page = metadata.get("page_number")
        if page:
            prefix.append(f"PAGE: {page}")
        content_type = metadata.get("content_type", "")
        if content_type:
            prefix.append(f"CONTENT TYPE: {content_type}")

    return "\n".join(prefix) + "\n\n" + text

In [6]:
enriched_texts  = []
enriched_chunks = []
boost_log       = Counter()

for chunk in chunks:
    doc_type   = detect_document_type(chunk)
    enriched   = enrich_text(chunk)
    chunk_type = chunk.get("metadata", {}).get("chunk_type", "unknown")

    # Always add once
    enriched_texts.append(enriched)
    enriched_chunks.append(chunk)
    boost_log[chunk_type] += 1

    # Boost WMS procedure chunks — these are the most query-critical
    if chunk_type == "wms_procedure":
        for _ in range(OPERATIONAL_BOOST - 1):
            enriched_texts.append(enriched)
            enriched_chunks.append(chunk)
            boost_log[f"{chunk_type}_boosted"] += 1

    # Boost schema overview chunks — critical for SQL generation
    if chunk_type == "schema_overview":
        enriched_texts.append(enriched)
        enriched_chunks.append(chunk)
        boost_log[f"{chunk_type}_boosted"] += 1

print(f"✅ Total embedding texts : {len(enriched_texts)}")
print(f"\n📊 Embedding breakdown:")
for k, v in boost_log.most_common():
    print(f"   {k:<40} : {v}")

✅ Total embedding texts : 673

📊 Embedding breakdown:
   text_prose                               : 470
   wms_procedure                            : 28
   wms_procedure_boosted                    : 28
   text_table                               : 25
   schema_overview                          : 18
   schema_overview_boosted                  : 18
   wms_overview                             : 18
   wms_join_logic                           : 18
   wms_safety_rules                         : 18
   schema_core_columns                      : 17
   schema_extra_columns                     : 15


In [7]:
print(f"🔄 Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

print(f"🔄 Embedding {len(enriched_texts)} texts...")
embeddings = model.encode(
    enriched_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

# L2 normalise for cosine similarity via inner product
norms      = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / np.clip(norms, 1e-10, None)

print(f"✅ Embeddings shape: {embeddings.shape}")
print(f"   Min norm (post-normalisation): {np.linalg.norm(embeddings, axis=1).min():.4f}")
print(f"   Max norm (post-normalisation): {np.linalg.norm(embeddings, axis=1).max():.4f}")

🔄 Loading model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1082.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Embedding 673 texts...


Batches: 100%|██████████| 22/22 [00:29<00:00,  1.35s/it]

✅ Embeddings shape: (673, 384)
   Min norm (post-normalisation): 1.0000
   Max norm (post-normalisation): 1.0000


In [8]:
dimension = embeddings.shape[1]

# IndexFlatIP = exact inner product search (correct for normalised vectors = cosine similarity)
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"✅ FAISS index built")
print(f"   Dimension : {dimension}")
print(f"   Vectors   : {index.ntotal}")

✅ FAISS index built
   Dimension : 384
   Vectors   : 673


In [9]:
faiss_path    = VECTOR_STORE / "faiss.index"
metadata_path = VECTOR_STORE / "metadata.pkl"
config_path   = VECTOR_STORE / "config.json"

# Save FAISS index
faiss.write_index(index, str(faiss_path))

# Save chunk metadata
with open(metadata_path, "wb") as f:
    pickle.dump(enriched_chunks, f)

# Save config so retriever knows what model & settings were used
config = {
    "model_name"        : MODEL_NAME,
    "total_vectors"     : index.ntotal,
    "dimension"         : dimension,
    "index_type"        : "IndexFlatIP",
    "normalised"        : True,
    "operational_boost" : OPERATIONAL_BOOST,
    "schema_boost"      : SCHEMA_BOOST,
    "chunk_type_counts" : dict(type_counts)
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"💾 Saved FAISS index     → {faiss_path}")
print(f"💾 Saved chunk metadata  → {metadata_path}")
print(f"💾 Saved config          → {config_path}")

💾 Saved FAISS index     → C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store\faiss.index
💾 Saved chunk metadata  → C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store\metadata.pkl
💾 Saved config          → C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store\config.json


In [ ]:
def sanity_check(query: str, top_k: int = 3):
    """Quick retrieval test to verify the vector store works correctly."""
    q_emb = model.encode([query], convert_to_numpy=True).astype("float32")
    q_emb = q_emb / np.clip(np.linalg.norm(q_emb, axis=1, keepdims=True), 1e-10, None)

    scores, indices = index.search(q_emb, top_k)

    print(f"\n🔎 Query: '{query}'")
    print(f"{'─'*60}")
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        chunk    = enriched_chunks[idx]
        meta     = chunk.get("metadata", {})
        print(f"  Rank {rank} | Score: {score:.4f}")
        print(f"    chunk_type : {meta.get('chunk_type', 'unknown')}")
        print(f"    source     : {meta.get('source', 'unknown')}")
        print(f"    category   : {meta.get('category', 'unknown')}")
        if meta.get("table_name"):
            print(f"    table      : {meta.get('table_name')}")
        if meta.get("procedure_name"):
            print(f"    procedure  : {meta.get('procedure_name')}")
        print(f"    text preview: {chunk['text'][:120].strip()}...")
        print()

# Test with different query types
sanity_check("What are the foreign keys in the reception header table?")
sanity_check("How do I check loading details for an order?")
sanity_check("What columns store reception date and time?")


🔎 Query: 'What are the foreign keys in the reception header table?'
────────────────────────────────────────────────────────────
  Rank 1 | Score: 0.5250
    chunk_type : schema_overview
    source     : REE_DAT.json
    category   : Database Tables
    table      : REE_DAT
    text preview: TABLE: REE_DAT
DESCRIPTION: RECEPTION HEADER TABLE
PRIMARY KEY: REE_KEYU
CATEGORY: Database Tables

FOREIGN KEY RELATION...

  Rank 2 | Score: 0.5250
    chunk_type : schema_overview
    source     : REE_DAT.json
    category   : Database Tables
    table      : REE_DAT
    text preview: TABLE: REE_DAT
DESCRIPTION: RECEPTION HEADER TABLE
PRIMARY KEY: REE_KEYU
CATEGORY: Database Tables

FOREIGN KEY RELATION...

  Rank 3 | Score: 0.5111
    chunk_type : schema_overview
    source     : CHG_DAT.json
    category   : Database Tables
    table      : CHG_DAT
    text preview: TABLE: CHG_DAT
DESCRIPTION: CONTAINER LOADING HEADER TABLE
PRIMARY KEY: CHG_KEYU
CATEGORY: Database Tables

FOREIGN KEY...


🔎 Q

: 